# LAB 10 — Fine-tuning & Controlled Experiments (Experiments 1–5)
Notebook huấn luyện mô hình `ExtractiveQAModel` trên SQuAD v1.1, đánh giá EM/F1, chạy ablation study về `max_length`, phân tích độ dài đáp án và xuất toàn bộ biểu đồ.

In [ ]:
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

from qa_model import (
    ExtractiveQAModel,
    keyword_overlap_baseline,
    compute_exact_match,
    compute_f1,
    decode_best_span,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Using device:", DEVICE)

In [ ]:
# Tải SQuAD v1.1
raw_datasets = load_dataset("rajpurkar/squad")
train_raw = raw_datasets["train"].shuffle(seed=SEED).select(range(15000))
val_raw = raw_datasets["validation"].shuffle(seed=SEED).select(range(2000))

# Đánh giá Heuristic Baseline (Mục 21)
baseline_ems, baseline_f1s = [], []
for ex in val_raw:
    pred = keyword_overlap_baseline(ex["question"], ex["context"])
    golds = ex["answers"]["text"]
    baseline_ems.append(max(compute_exact_match(pred, g) for g in golds))
    baseline_f1s.append(max(compute_f1(pred, g) for g in golds))

print(f"Heuristic Baseline -> EM: {np.mean(baseline_ems)*100:.2f}% | F1: {np.mean(baseline_f1s)*100:.2f}%")

In [ ]:
def prepare_train_features(examples, max_length=384, stride=128):
    questions = [q.strip() for q in examples["question"]]
    tokenized_examples = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")
    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized_examples.sequence_ids(i)
        answers = examples["answers"][sample_mapping[i]]

        if len(answers["answer_start"]) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
            continue

        answer_start = answers["answer_start"][0]
        answer_end = answer_start + len(answers["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1
        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if not (offsets[token_start_index][0] <= answer_start and offsets[token_end_index][1] >= answer_end):
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= answer_start:
                token_start_index += 1
            start_token = token_start_index - 1
            while offsets[token_end_index][1] >= answer_end:
                token_end_index -= 1
            end_token = token_end_index + 1
            tokenized_examples["start_positions"].append(start_token)
            tokenized_examples["end_positions"].append(end_token)

    return tokenized_examples


def evaluate_model(model, dataset_split, max_length=384, stride=128):
    model.eval()
    records = []
    with torch.no_grad():
        for ex in dataset_split:
            enc = tokenizer(
                ex["question"].strip(),
                ex["context"],
                truncation="only_second",
                max_length=max_length,
                stride=stride,
                return_overflowing_tokens=True,
                return_offsets_mapping=True,
                padding="max_length",
                return_tensors="pt",
            )
            offsets_all = enc.pop("offset_mapping")
            enc.pop("overflow_to_sample_mapping")

            best_pred, best_score = "", -1e9
            out = model(enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE))
            for w in range(enc["input_ids"].size(0)):
                pred_w, score_w = decode_best_span(
                    out["start_logits"][w],
                    out["end_logits"][w],
                    offsets_all[w].tolist(),
                    enc.sequence_ids(w),
                    ex["context"],
                )
                if score_w > best_score:
                    best_score = score_w
                    best_pred = pred_w

            golds = ex["answers"]["text"]
            em = max(compute_exact_match(best_pred, g) for g in golds)
            f1 = max(compute_f1(best_pred, g) for g in golds)
            gold_len = len(tokenizer.tokenize(golds[0]))
            records.append({
                "question": ex["question"],
                "gold": golds[0],
                "prediction": best_pred,
                "em": em,
                "f1": f1,
                "gold_len": gold_len,
            })
    return pd.DataFrame(records)

In [ ]:
def train_qa(max_length=384, stride=128, epochs=3, batch_size=16, lr=3e-5):
    train_ds = train_raw.map(
        lambda x: prepare_train_features(x, max_length=max_length, stride=stride),
        batched=True,
        remove_columns=train_raw.column_names,
    )
    train_ds.set_format("torch")
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = ExtractiveQAModel(MODEL_NAME).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                out = model(
                    input_ids=batch["input_ids"].to(DEVICE),
                    attention_mask=batch["attention_mask"].to(DEVICE),
                    start_positions=batch["start_positions"].to(DEVICE),
                    end_positions=batch["end_positions"].to(DEVICE),
                )
                loss = out["loss"]
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += loss.item()

        eval_df = evaluate_model(model, val_raw, max_length=max_length, stride=stride)
        print(
            f"[max_len={max_length}] Epoch {epoch}/{epochs} | "
            f"Train Loss: {running_loss/len(train_loader):.4f} | "
            f"Val EM: {eval_df['em'].mean()*100:.2f}% | Val F1: {eval_df['f1'].mean()*100:.2f}%"
        )

    elapsed = time.time() - t0
    return model, eval_df, elapsed
